In [8]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [9]:
#  1. Create Spark Session

spark = (
    SparkSession.builder
    .appName("Zomato Daily Aggregation Job")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")


In [10]:
# 2. Read Raw CSV (Week 1 dataset)

zomato_path = "/kaggle/input/datasets/damanthloki/zomato/zomato_dataset.csv"   # change if local

df = (
    spark.read
    .option("header", True)
    .option("multiLine", True)   # IMPORTANT (reviews_list has new lines)
    .option("quote", '"')
    .option("escape", '"')
    .option("mode", "PERMISSIVE")
    .option("inferSchema", False)  # avoid wrong datatype inference
    .csv(zomato_path)
)

print("Raw Schema:")
df.printSchema()


Raw Schema:
root
 |-- url: string (nullable = true)
 |-- address: string (nullable = true)
 |-- name: string (nullable = true)
 |-- online_order: string (nullable = true)
 |-- book_table: string (nullable = true)
 |-- rate: string (nullable = true)
 |-- votes: string (nullable = true)
 |-- phone: string (nullable = true)
 |-- location: string (nullable = true)
 |-- rest_type: string (nullable = true)
 |-- dish_liked: string (nullable = true)
 |-- cuisines: string (nullable = true)
 |-- approx_cost(for two people): string (nullable = true)
 |-- reviews_list: string (nullable = true)
 |-- menu_item: string (nullable = true)
 |-- listed_in(type): string (nullable = true)
 |-- listed_in(city): string (nullable = true)



In [11]:
# 3. Basic Cleaning

df_clean = (
    df
    # ---------- Rating cleanup ----------
    .withColumn(
        "rating_tmp",
        regexp_replace(col("rate"), "/5", "")
    )
    .withColumn(
        "rating",
        when(col("rating_tmp").isin("NEW", "-", "null"), None)
        .otherwise(expr("try_cast(rating_tmp as double)"))
    )

    # ---------- Cost cleanup ----------
    .withColumn(
        "cost_tmp",
        regexp_replace(
            col("approx_cost(for two people)"), ",", ""
        )
    )
    .withColumn(
        "cost_for_two",
        expr("try_cast(cost_tmp as double)")
    )

    # ---------- Votes cleanup ----------
    .withColumn(
        "votes",
        expr("try_cast(votes as int)")
    )
)

# Add ingestion date (simulate daily pipeline)
df_clean = df_clean.withColumn(
    "ingestion_date",
    current_date()
)


In [12]:
# 4. Read Dimension Mapping (Zone Lookup)

zone_lookup_path = "/kaggle/input/datasets/damanthloki/zomato/zone_lookup.csv"

zone_df = (
    spark.read
    .option("header", True)
    .csv(zone_lookup_path)
)

print("Zone Lookup:")
zone_df.show()

Zone Lookup:
+-----------+-------+
|   location|   zone|
+-----------+-------+
|        BTM|  South|
|Indiranagar|   East|
| Whitefield|   East|
|        HSR|  South|
|Koramangala|  South|
|    MG Road|Central|
|  Jayanagar|  South|
+-----------+-------+



In [13]:
# 5. Join with Dimension Table
# (Broadcast because small table)

from pyspark.sql.functions import broadcast

df_joined = (
    df_clean.join(
        broadcast(zone_df),
        on="location",
        how="left"
    )
)

In [14]:
# 6. Compute Daily Aggregates

# Metrics:
# - avg rating
# - median rating
# - avg cost
# - total votes
# grouped by zone + day

daily_agg = (
    df_joined.groupBy("ingestion_date", "zone")
    .agg(
        avg("rating").alias("avg_rating"),
        expr("percentile_approx(rating, 0.5)").alias("median_rating"),
        avg("cost_for_two").alias("avg_cost_for_two"),
        sum("votes").alias("total_votes"),
        count("*").alias("restaurant_count")
    )
)

print("Daily Aggregates:")
daily_agg.show(truncate=False)

Daily Aggregates:


+--------------+-------+------------------+-------------+------------------+-----------+----------------+
|ingestion_date|zone   |avg_rating        |median_rating|avg_cost_for_two  |total_votes|restaurant_count|
+--------------+-------+------------------+-------------+------------------+-----------+----------------+
|2026-02-27    |East   |3.73286672499271  |3.8          |625.3368496763366 |1662836    |4227            |
|2026-02-27    |NULL   |3.70668724625038  |3.7          |564.7249142483803 |10964098   |36951           |
|2026-02-27    |South  |3.6449357124114448|3.7          |433.2417467613874 |1612940    |9621            |
|2026-02-27    |Central|3.855856966707763 |4.0          |1155.7046979865772|432111     |918             |
+--------------+-------+------------------+-------------+------------------+-----------+----------------+



In [15]:
# 7. Write Partitioned Parquet Output

output_path = "/kaggle/working/zomato_parquet_output"

(
    daily_agg.write
    .mode("overwrite")
    .partitionBy("ingestion_date", "zone")
    .parquet(output_path)
)

print(" Partitioned Parquet written successfully!")

 Partitioned Parquet written successfully!


In [16]:
# 8. Verify Output

result = spark.read.parquet(output_path)
result.show()

spark.stop()

+------------------+-------------+------------------+-----------+----------------+--------------+-------+
|        avg_rating|median_rating|  avg_cost_for_two|total_votes|restaurant_count|ingestion_date|   zone|
+------------------+-------------+------------------+-----------+----------------+--------------+-------+
| 3.855856966707763|          4.0|1155.7046979865772|     432111|             918|    2026-02-27|Central|
|3.6449357124114448|          3.7| 433.2417467613874|    1612940|            9621|    2026-02-27|  South|
|  3.73286672499271|          3.8| 625.3368496763366|    1662836|            4227|    2026-02-27|   East|
|  3.70668724625038|          3.7| 564.7249142483803|   10964098|           36951|    2026-02-27|   NULL|
+------------------+-------------+------------------+-----------+----------------+--------------+-------+



In [19]:
import shutil

shutil.make_archive(
    "/kaggle/working/zomato_parquet_output",
    'zip',
    "/kaggle/working/zomato_parquet_output"
)

print("ZIP file created!")

ZIP file created!
